# V11 Oscar — Pioneer Edition

## Identical to V11 Oscar (ICBHI-only) with one change:
## - `class_names` is built dynamically from `label_map` instead of hardcoded
## - `num_classes` automatically adapts to however many classes preprocessing produced
## - Everything else — architecture, training, evaluation — is unchanged


## 1. Imports and Reproducibility

In [ ]:
import os
import random
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)

# ============================================================
# V11 Oscar — 3 targeted changes from Lima baseline
# ============================================================
#
# 1. TARGET_COUNT raised from 500 to 700
#    → lets genuine test recordings contribute to fold training
#    → Lima's cap=500 effectively discarded test[tr_idx] data
#
# 2. SpecAugment added on MFCC/Chroma features during training
#    → frequency + time masking, complementary to audio augmentation
#    → expected to close the 14pp CV-to-unseen F1 gap
#
# 3. Everything else UNCHANGED from Lima:
#    → uniform FocalLoss gamma=1.5 (not per-class)
#    → classifier 64→128→6 (not funnel)
#    → fuse_dim=64, single-head TemporalAttention
#    → same CNN, LSTM, GatedFusion, optimizer, scheduler
# ============================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## 2. Paths and Feature Parameters

In [ ]:
import numpy as np
import pandas as pd
import librosa as lb

BASE_DIR  = "../../dataset"

TRAIN_DIR = os.path.join(BASE_DIR, "train-segments")
TEST_DIR  = os.path.join(BASE_DIR, "test-segments")
VAL_DIR   = os.path.join(BASE_DIR, "val-segments")

TRAIN_CSV = os.path.join(BASE_DIR, "train_segments.csv")
TEST_CSV  = os.path.join(BASE_DIR, "test_segments.csv")
VAL_CSV   = os.path.join(BASE_DIR, "val_segments.csv")

SR              = 4000
N_MFCC          = 40
TARGET_MFCC_T   = 679
TARGET_CHROMA_T = 259

## 3. Feature Extraction Helpers

In [ ]:
def extract_mfcc_and_chroma(y, sr):
    mfcc   = lb.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)
    chroma = lb.feature.chroma_stft(y=y, sr=sr)
    return mfcc, chroma


def pad_or_crop(feat, target_T):
    F, T = feat.shape
    if T > target_T:
        return feat[:, :target_T]
    if T < target_T:
        return np.pad(feat, ((0, 0), (0, target_T - T)), mode="constant")
    return feat

## 4. Build Features from CSV

In [ ]:
def build_features_from_csv(csv_path, audio_dir):
    df        = pd.read_csv(csv_path)
    mfcc_list = []
    chroma_list = []
    labels    = []

    diseases  = sorted(df["disease"].unique())
    label_map = {d: i for i, d in enumerate(diseases)}

    for _, row in df.iterrows():
        filename  = os.path.basename(row["segment_file"])
        label     = row["disease"]
        file_path = os.path.join(audio_dir, filename)

        if not os.path.exists(file_path):
            raise FileNotFoundError(f"Missing file: {file_path}")

        y, sr        = lb.load(file_path, sr=SR)
        mfcc, chroma = extract_mfcc_and_chroma(y, sr)
        mfcc         = pad_or_crop(mfcc,   TARGET_MFCC_T)
        chroma       = pad_or_crop(chroma, TARGET_CHROMA_T)

        mfcc_list.append(mfcc)
        chroma_list.append(chroma)
        labels.append(label_map[label])

    return np.stack(mfcc_list), np.stack(chroma_list), np.array(labels), label_map

## 5. Load Data

In [ ]:
X_mfcc_train, X_chroma_train, y_train, label_map = build_features_from_csv(TRAIN_CSV, TRAIN_DIR)
X_mfcc_test,  X_chroma_test,  y_test,  _         = build_features_from_csv(TEST_CSV,  TEST_DIR)
X_mfcc_val,   X_chroma_val,   y_val,   _         = build_features_from_csv(VAL_CSV,   VAL_DIR)

## 6. Convert to Tensors

In [ ]:
X_mfcc_train   = torch.from_numpy(X_mfcc_train).float()
X_chroma_train = torch.from_numpy(X_chroma_train).float()
y_train        = torch.from_numpy(y_train).long()

X_mfcc_test    = torch.from_numpy(X_mfcc_test).float()
X_chroma_test  = torch.from_numpy(X_chroma_test).float()
y_test         = torch.from_numpy(y_test).long()

X_mfcc_unseen   = torch.from_numpy(X_mfcc_val).float()
X_chroma_unseen = torch.from_numpy(X_chroma_val).float()
y_unseen        = torch.from_numpy(y_val).long()

num_classes = len(label_map)
print("Train:",  X_mfcc_train.shape,  X_chroma_train.shape,  y_train.shape)
print("Test: ",  X_mfcc_test.shape,   X_chroma_test.shape,   y_test.shape)
print("Unseen:", X_mfcc_unseen.shape, X_chroma_unseen.shape, y_unseen.shape)
print("Label map:", label_map)

## 7. Dataset and DataLoader

In [ ]:
# -------------------------------------------------------
# SpecAugment — feature-level augmentation  [NEW in Oscar]
#
# Preprocessing already augments raw audio (noise, pitch, speed, gain).
# SpecAugment works at the FEATURE level — masking random frequency
# bands and time bands on MFCC/Chroma spectrograms during training.
# These are complementary: audio augmentation creates new recordings,
# SpecAugment teaches the model to be robust to missing spectral info.
#
# Applied only during training (controlled by augment=True flag).
# -------------------------------------------------------
def spec_augment(feat, freq_mask_width=4, time_mask_width=25, num_masks=2):
    """
    Apply SpecAugment to a 2D feature array [F, T].
    - freq_mask_width: max number of frequency bins to mask
    - time_mask_width: max number of time frames to mask
    - num_masks: number of masks per dimension
    """
    feat = feat.clone()
    F, T = feat.shape

    for _ in range(num_masks):
        # Frequency mask
        f = torch.randint(0, freq_mask_width + 1, (1,)).item()
        f0 = torch.randint(0, max(1, F - f), (1,)).item()
        feat[f0:f0 + f, :] = 0

        # Time mask
        t = torch.randint(0, time_mask_width + 1, (1,)).item()
        t0 = torch.randint(0, max(1, T - t), (1,)).item()
        feat[:, t0:t0 + t] = 0

    return feat


class DualFeatureDataset(Dataset):
    """
    Oscar adds SpecAugment support via augment flag.
    When augment=True (training), applies spec_augment to both MFCC and Chroma.
    When augment=False (validation/unseen), returns features unchanged.
    """
    def __init__(self, X_mfcc, X_chroma, y, augment=False):
        assert len(X_mfcc) == len(X_chroma) == len(y)
        self.X_mfcc   = X_mfcc
        self.X_chroma = X_chroma
        self.y        = y
        self.augment  = augment

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        mfcc   = self.X_mfcc[idx]
        chroma = self.X_chroma[idx]

        if self.augment:
            mfcc   = spec_augment(mfcc,   freq_mask_width=4,  time_mask_width=25, num_masks=2)
            chroma = spec_augment(chroma, freq_mask_width=2,  time_mask_width=15, num_masks=2)

        return mfcc, chroma, self.y[idx]


# Unseen loader — batch_size=32 for evaluation only (no augmentation)
unseen_loader = DataLoader(
    DualFeatureDataset(X_mfcc_unseen, X_chroma_unseen, y_unseen, augment=False),
    batch_size=32, shuffle=False, num_workers=0, pin_memory=True
)


## 8. Model Architecture

### Changes from Echo

**1. TemporalAttention added to both MFCC and Chroma LSTM pathways.**
Echo used `x[:, -1, :]` — the last hidden state only — discarding all temporal information
from earlier time steps. TemporalAttention assigns a learned weight to every time step
and returns a weighted sum, letting the model focus on the most diagnostically informative
frames within the 3-second window. Proven in Charlie (best CV F1 = 0.5343).

**2. LayerNorm added after TemporalAttention on both LSTM pathways.**
Stabilises the output distribution of the LSTM before it enters the fusion gate.
Without normalisation, large magnitude differences between CNN and LSTM vectors
can bias the gate even before it learns.

**3. Chroma LSTM inter-layer dropout fixed.**
Echo had `self.mfcc_lstm_drop` applied between MFCC LSTM layers but the Chroma
LSTM pathway was missing the dropout application in the forward pass — an
inconsistency. Both pathways now apply inter-layer dropout symmetrically.

**4. GatedFusion gate constraint kept at [0.35, 0.65] from Echo.**
The constrained gate is what produced Echo's best unseen result. It prevents
either CNN or LSTM from being suppressed below 35%, ensuring both pathways
contribute to the final representation.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class GatedFusion(nn.Module):
    """
    Constrained gated fusion — weight w clamped to [0.35, 0.65].
    Unchanged from Lima. All 3 machines converged to the same stable
    gate pattern in Lima and November, confirming the constraint
    produces a reproducible learned fusion equilibrium.
    """
    def __init__(self, dim: int, hidden: int = 128, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim * 2, hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden, dim),
        )

    def forward(self, a: torch.Tensor, b: torch.Tensor):
        # Constrained sigmoid — clamped to [0.35, 0.65]
        w     = torch.clamp(
            torch.sigmoid(self.net(torch.cat([a, b], dim=1))),
            min=0.35, max=0.65
        )
        fused = w * a + (1.0 - w) * b
        return fused, w


class TemporalAttention(nn.Module):
    """
    Single-head learned attention over LSTM time steps.
    Unchanged from Lima — Mike's 4-head version was a confirmed regression.

    Assigns a learned scalar weight to each time step and returns
    their weighted sum, letting the model focus on diagnostically
    informative frames in the 3-second respiratory window.
    """
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, T, H]
        scores  = self.attn(x).squeeze(-1)            # [B, T]
        weights = torch.softmax(scores, dim=1)        # [B, T]
        out     = (weights.unsqueeze(-1) * x).sum(1)  # [B, H]
        return out


class DualChannelCNNLSTM_WithAttentionFusion(nn.Module):

    def __init__(self, num_classes: int = 6, dropout_p: float = 0.4, fuse_dim: int = 64):
        super().__init__()
        self.fuse_dim = fuse_dim

        # -------------------------------------------------------
        # MFCC CNN — unchanged from Lima
        # -------------------------------------------------------
        self.mfcc_cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(0.2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(0.2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(0.2),
        )

        # -------------------------------------------------------
        # Chroma CNN — unchanged from Lima
        # -------------------------------------------------------
        self.chroma_cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(0.2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(0.2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(0.2),
        )

        self.gap2d           = nn.AdaptiveAvgPool2d((1, 1))
        self.mfcc_cnn_proj   = nn.Linear(128, fuse_dim)
        self.chroma_cnn_proj = nn.Linear(128, fuse_dim)

        # -------------------------------------------------------
        # MFCC LSTM — 2 layers + inter-layer Dropout(0.3)
        #           + TemporalAttention + LayerNorm  [NEW in Lima]
        #
        # Echo used x[:, -1, :] as LSTM readout (last hidden state only).
        # Lima replaces this with TemporalAttention over all time steps,
        # followed by LayerNorm to stabilise the distribution before fusion.
        # -------------------------------------------------------
        self.mfcc_lstm1     = nn.LSTM(40,       128,      batch_first=True)
        self.mfcc_lstm_drop = nn.Dropout(p=0.3)
        self.mfcc_lstm2     = nn.LSTM(128,      fuse_dim, batch_first=True)
        self.mfcc_attn      = TemporalAttention(fuse_dim)   # NEW
        self.mfcc_lstm_norm = nn.LayerNorm(fuse_dim)        # NEW

        # -------------------------------------------------------
        # CHROMA LSTM — 2 layers + inter-layer Dropout(0.3)
        #             + TemporalAttention + LayerNorm  [NEW in Lima]
        #
        # Echo defined self.chroma_lstm_drop but never applied it in
        # the forward pass — an inconsistency vs the MFCC pathway.
        # Lima applies chroma_lstm_drop symmetrically between layers.
        # -------------------------------------------------------
        self.chroma_lstm1     = nn.LSTM(12,      128,      batch_first=True)
        self.chroma_lstm_drop = nn.Dropout(p=0.3)
        self.chroma_lstm2     = nn.LSTM(128,     fuse_dim, batch_first=True)
        self.chroma_attn      = TemporalAttention(fuse_dim)  # NEW
        self.chroma_lstm_norm = nn.LayerNorm(fuse_dim)       # NEW

        # -------------------------------------------------------
        # Fusion modules — constrained GatedFusion (unchanged from Lima)
        # fuse_cnn_modality  : MFCC-CNN  vs Chroma-CNN  (intra-CNN)
        # fuse_lstm_modality : MFCC-LSTM vs Chroma-LSTM (intra-LSTM)
        # fuse_extractor     : CNN-fused vs LSTM-fused  (inter-modal)
        # -------------------------------------------------------
        self.fuse_cnn_modality  = GatedFusion(dim=fuse_dim, hidden=128, dropout=0.1)
        self.fuse_lstm_modality = GatedFusion(dim=fuse_dim, hidden=128, dropout=0.1)
        self.fuse_extractor     = GatedFusion(dim=fuse_dim, hidden=128, dropout=0.1)

        # -------------------------------------------------------
        # Classifier head — 64→128→6 (unchanged from Lima)
        # November's funnel (64→32→6) was a potential contributor to
        # its 3.7pp regression. Lima's "trumpet" kept here.
        # -------------------------------------------------------
        self.fc1     = nn.Linear(fuse_dim, 128)
        self.dropout = nn.Dropout(p=dropout_p)
        self.fc_out  = nn.Linear(128, num_classes)

    def forward(self, mfcc: torch.Tensor, chroma: torch.Tensor, return_attn: bool = False):

        # --- CNN pathway (unchanged from Lima) ---
        mfcc_cnn_vec   = self.gap2d(self.mfcc_cnn(mfcc.unsqueeze(1))).flatten(1)
        mfcc_cnn_vec   = self.mfcc_cnn_proj(mfcc_cnn_vec)

        chroma_cnn_vec = self.gap2d(self.chroma_cnn(chroma.unsqueeze(1))).flatten(1)
        chroma_cnn_vec = self.chroma_cnn_proj(chroma_cnn_vec)

        # --- MFCC LSTM pathway ---
        # Lima: TemporalAttention + LayerNorm replaces x[:, -1, :]
        x, _ = self.mfcc_lstm1(mfcc.transpose(1, 2))   # [B, T, 128]
        x     = self.mfcc_lstm_drop(x)
        x, _ = self.mfcc_lstm2(x)                      # [B, T, fuse_dim]
        mfcc_lstm_vec = self.mfcc_lstm_norm(self.mfcc_attn(x))  # [B, fuse_dim]

        # --- Chroma LSTM pathway ---
        # Lima: chroma_lstm_drop now applied (was missing in Echo)
        # Lima: TemporalAttention + LayerNorm replaces y[:, -1, :]
        y, _ = self.chroma_lstm1(chroma.transpose(1, 2))  # [B, T, 128]
        y     = self.chroma_lstm_drop(y)                  # FIXED — was missing in Echo
        y, _ = self.chroma_lstm2(y)                       # [B, T, fuse_dim]
        chroma_lstm_vec = self.chroma_lstm_norm(self.chroma_attn(y))  # [B, fuse_dim]

        # --- Intra-modal fusion (unchanged from Lima) ---
        cnn_fused,  w_cnn  = self.fuse_cnn_modality(mfcc_cnn_vec,   chroma_cnn_vec)
        lstm_fused, w_lstm = self.fuse_lstm_modality(mfcc_lstm_vec, chroma_lstm_vec)

        # --- Inter-modal fusion (unchanged from Lima) ---
        final_fused, w_inter = self.fuse_extractor(cnn_fused, lstm_fused)

        # --- Classifier (unchanged from Lima) ---
        z      = F.relu(self.fc1(final_fused))
        z      = self.dropout(z)
        logits = self.fc_out(z)

        if return_attn:
            return logits, {
                "w_cnn_mean":   float(w_cnn.mean().item()),
                "w_lstm_mean":  float(w_lstm.mean().item()),
                "w_inter_mean": float(w_inter.mean().item()),
            }
        return logits

## 9. Early Stopping

In [ ]:
@dataclass
class EarlyStopping:
    patience:       int   = 7
    min_delta:      float = 0.0
    best:           float = float("inf")
    num_bad_epochs: int   = 0
    stopped:        bool  = False

    def step(self, current: float) -> bool:
        improved = (self.best - current) > self.min_delta
        if improved:
            self.best           = current
            self.num_bad_epochs = 0
        else:
            self.num_bad_epochs += 1
            if self.num_bad_epochs >= self.patience:
                self.stopped = True
        return self.stopped


checkpoint_path = "best_model.pt"

## 10. Training Loop (run_epoch)

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, balanced_accuracy_score


def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)

    all_losses, all_preds, all_targets = [], [], []
    attn_w_cnn, attn_w_lstm, attn_w_inter = [], [], []

    for mfcc, chroma, y in loader:
        mfcc   = mfcc.to(device,   non_blocking=True)
        chroma = chroma.to(device, non_blocking=True)
        y      = y.to(device,      non_blocking=True)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_train):
            logits, attn = model(mfcc, chroma, return_attn=True)
            loss = criterion(logits, y)

        if is_train:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        all_losses.append(loss.item())
        preds = torch.argmax(logits, dim=1)
        all_preds.append(preds.detach().cpu().numpy())
        all_targets.append(y.detach().cpu().numpy())
        attn_w_cnn.append(attn["w_cnn_mean"])
        attn_w_lstm.append(attn["w_lstm_mean"])
        attn_w_inter.append(attn["w_inter_mean"])

    all_preds   = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)

    acc     = accuracy_score(all_targets, all_preds)
    bal_acc = balanced_accuracy_score(all_targets, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(
        all_targets, all_preds, average="macro", zero_division=0
    )

    return (
        float(np.mean(all_losses)),
        acc, bal_acc, prec, rec, f1,
        float(np.mean(attn_w_cnn)),
        float(np.mean(attn_w_lstm)),
        float(np.mean(attn_w_inter)),
    )

## 11. 10-Fold Cross-Validation

### Training configuration

All hyperparameters below are **identical to Echo** except Focal Loss γ (2.0 → 1.5):

- **FocalLoss γ=1.5** — softened from Echo's 2.0. γ=2.0 was too aggressive and hurt
  Bronchiolitis recall by ~21pp (confirmed across Charlie and Golf). γ=1.5 keeps the
  minority-class focus without over-penalising classes the model was already learning.
- **AdamW** lr=1e-4, weight_decay=1e-4 — unchanged from Echo
- **ReduceLROnPlateau** factor=0.5, patience=6 — unchanged from Echo.
  Proven better than CosineAnnealingWarmRestarts for unseen generalisation:
  responds directly to validation loss rather than cycling blindly.
- **batch_size=64** — unchanged from Echo
- **patience=25** — unchanged from Echo
- **TARGET_COUNT=500** — unchanged from Echo

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix
import seaborn as sns

epochs       = 500
n_splits     = 10
batch_size   = 64
TARGET_COUNT = 700

# -------------------------------------------------------------------
# Focal Loss  gamma=1.5  [UNCHANGED from Lima]
# Lima's uniform gamma=1.5 is proven — produced best unseen F1 (0.3890)
# across all versions. Mike/November's per-class gammas were net negative.
# Retained without modification.
# -------------------------------------------------------------------
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=1.5):
        super().__init__()
        self.weight = weight
        self.gamma  = gamma

    def forward(self, logits, targets):
        log_prob      = F.log_softmax(logits, dim=1)
        prob          = torch.exp(log_prob)
        log_prob_true = log_prob.gather(1, targets.unsqueeze(1)).squeeze(1)
        prob_true     = prob.gather(1, targets.unsqueeze(1)).squeeze(1)
        focal_weight  = (1.0 - prob_true) ** self.gamma
        loss          = -focal_weight * log_prob_true
        if self.weight is not None:
            loss = loss * self.weight[targets]
        return loss.mean()


# -------------------------------------------------------------------
# Cap each class to TARGET_COUNT=700 samples per fold  [CHANGED from Lima's 500]
#
# Lima capped at 500, which effectively discarded most test[tr_idx] data
# since augmented train already had 500/class. At cap=700, genuine test
# recordings (from different patients) can contribute to training while
# keeping COPD:minority ratio at ~1.4:1 (manageable with focal loss).
# -------------------------------------------------------------------
def cap_per_class(mfcc, chroma, y, cap=TARGET_COUNT, seed=SEED):
    rng      = np.random.default_rng(seed)
    keep_idx = []
    for cls in np.unique(y):
        cls_idx = np.where(y == cls)[0]
        if len(cls_idx) > cap:
            cls_idx = rng.choice(cls_idx, cap, replace=False)
        keep_idx.append(cls_idx)
    keep_idx = np.sort(np.concatenate(keep_idx))
    return mfcc[keep_idx], chroma[keep_idx], y[keep_idx]


# -------------------------------------------------------------------
# Convert to numpy for K-Fold indexing
# -------------------------------------------------------------------
X_mfcc_train_np   = X_mfcc_train.numpy()   if isinstance(X_mfcc_train,   torch.Tensor) else X_mfcc_train
X_chroma_train_np = X_chroma_train.numpy() if isinstance(X_chroma_train, torch.Tensor) else X_chroma_train
y_train_np        = y_train.numpy()        if isinstance(y_train,        torch.Tensor) else y_train

X_mfcc_test_np    = X_mfcc_test.numpy()    if isinstance(X_mfcc_test,    torch.Tensor) else X_mfcc_test
X_chroma_test_np  = X_chroma_test.numpy()  if isinstance(X_chroma_test,  torch.Tensor) else X_chroma_test
y_test_np         = y_test.numpy()         if isinstance(y_test,         torch.Tensor) else y_test

# -------------------------------------------------------------------
# K-Fold on raw test indices only
# -------------------------------------------------------------------
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)

fold_summaries  = []
best_fold_loss  = float("inf")
best_fold_state = None
all_cv_preds    = []
all_cv_targets  = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(y_test_np)), y_test_np), start=1):
    print(f"\n========== Fold {fold}/{n_splits} ==========")

    # Val fold = raw test slice — imbalanced, never augmented
    fold_val_mfcc   = X_mfcc_test_np[va_idx]
    fold_val_chroma = X_chroma_test_np[va_idx]
    fold_val_y      = y_test_np[va_idx]

    # Train fold = aug train + raw test train slice, capped per class
    fold_train_mfcc_raw   = np.concatenate([X_mfcc_train_np,   X_mfcc_test_np[tr_idx]],   axis=0)
    fold_train_chroma_raw = np.concatenate([X_chroma_train_np, X_chroma_test_np[tr_idx]], axis=0)
    fold_train_y_raw      = np.concatenate([y_train_np,        y_test_np[tr_idx]],         axis=0)

    fold_train_mfcc, fold_train_chroma, fold_train_y = cap_per_class(
        fold_train_mfcc_raw, fold_train_chroma_raw, fold_train_y_raw, cap=TARGET_COUNT
    )

    print(f"  Train: {len(fold_train_y)} segs | Val: {len(fold_val_y)} segs (raw/imbalanced)")
    for cls in np.unique(fold_train_y):
        print(f"    class {cls}: {(fold_train_y==cls).sum()} train | {(fold_val_y==cls).sum()} val")

    fold_train_loader = DataLoader(
        DualFeatureDataset(
            torch.from_numpy(fold_train_mfcc).float(),
            torch.from_numpy(fold_train_chroma).float(),
            torch.from_numpy(fold_train_y).long(),
            augment=True,  # SpecAugment ON for training  [NEW in Oscar]
        ),
        batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True
    )
    fold_val_loader = DataLoader(
        DualFeatureDataset(
            torch.from_numpy(fold_val_mfcc).float(),
            torch.from_numpy(fold_val_chroma).float(),
            torch.from_numpy(fold_val_y).long(),
            augment=False,  # SpecAugment OFF for validation  [NEW in Oscar]
        ),
        batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True
    )

    # Fresh model per fold — dropout_p=0.4, fuse_dim=64 unchanged from Lima
    model = DualChannelCNNLSTM_WithAttentionFusion(
        num_classes=num_classes,
        dropout_p=0.4,
        fuse_dim=64,
    ).to(device)

    fold_class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.arange(num_classes),
        y=fold_train_y
    )
    fold_class_weights = torch.tensor(fold_class_weights, dtype=torch.float32).to(device)

    # gamma=1.5 — unchanged from Lima (proven best)
    criterion = FocalLoss(weight=fold_class_weights, gamma=1.5)

    # lr=1e-4, weight_decay=1e-4 — unchanged from Lima
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

    # ReduceLROnPlateau — unchanged from Lima
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=6, min_lr=1e-6
    )

    # patience=25 — unchanged from Lima
    early = EarlyStopping(patience=25, min_delta=1e-4)

    history = {
        "train_loss": [], "train_acc": [], "train_f1": [],
        "val_loss":   [], "val_acc":   [], "val_f1":   [],
        "lr":         [],
        "train_w_cnn": [], "train_w_lstm": [], "train_w_inter": [],
        "val_w_cnn":   [], "val_w_lstm":   [], "val_w_inter":   [],
    }

    best_this_fold       = float("inf")
    best_state_this_fold = None

    for epoch in range(1, epochs + 1):
        (train_loss, train_acc, train_bal_acc, train_prec, train_rec, train_f1,
         train_w_cnn, train_w_lstm, train_w_inter) = run_epoch(model, fold_train_loader, optimizer=optimizer)

        (val_loss, val_acc, val_bal_acc, val_prec, val_rec, val_f1,
         val_w_cnn, val_w_lstm, val_w_inter) = run_epoch(model, fold_val_loader, optimizer=None)

        scheduler.step(val_loss)
        lr = optimizer.param_groups[0]["lr"]

        history["lr"].append(lr)
        history["train_loss"].append(train_loss);     history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc);       history["val_acc"].append(val_acc)
        history["train_f1"].append(train_f1);         history["val_f1"].append(val_f1)
        history["train_w_cnn"].append(train_w_cnn);   history["val_w_cnn"].append(val_w_cnn)
        history["train_w_lstm"].append(train_w_lstm);  history["val_w_lstm"].append(val_w_lstm)
        history["train_w_inter"].append(train_w_inter); history["val_w_inter"].append(val_w_inter)

        print(
            f"Fold {fold} | Epoch {epoch:03d} | "
            f"train loss {train_loss:.4f} acc {train_acc:.4f} "
            f"bal_acc {train_bal_acc:.4f} f1 {train_f1:.4f} | "
            f"val loss {val_loss:.4f} acc {val_acc:.4f} "
            f"bal_acc {val_bal_acc:.4f} f1 {val_f1:.4f} | "
            f"attn(cnn={val_w_cnn:.3f}, lstm={val_w_lstm:.3f}, inter={val_w_inter:.3f}) | "
            f"lr {lr:.2e}"
        )

        if val_loss <= best_this_fold - early.min_delta:
            best_this_fold       = val_loss
            best_state_this_fold = {k: v.detach().cpu().clone()
                                    for k, v in model.state_dict().items()}

        if early.step(val_loss):
            print("Early stopping triggered.")
            break

    model.load_state_dict(best_state_this_fold)
    model.eval()

    fold_preds, fold_targets = [], []
    with torch.no_grad():
        for mfcc, chroma, y in fold_val_loader:
            logits = model(mfcc.to(device), chroma.to(device))
            fold_preds.append(torch.argmax(logits, dim=1).cpu().numpy())
            fold_targets.append(y.numpy())

    fold_preds   = np.concatenate(fold_preds)
    fold_targets = np.concatenate(fold_targets)
    all_cv_preds.extend(fold_preds)
    all_cv_targets.extend(fold_targets)

    fold_summaries.append({
        "fold":          fold,
        "best_val_loss": best_this_fold,
        "best_val_acc":  max(history["val_acc"]) if history["val_acc"] else float("nan"),
        "best_val_f1":   max(history["val_f1"])  if history["val_f1"]  else float("nan"),
        "history":       history,
    })

    if best_this_fold < best_fold_loss and best_state_this_fold is not None:
        best_fold_loss  = best_this_fold
        best_fold_state = best_state_this_fold


# -------------------------------------------------------------------
# CV Summary
# -------------------------------------------------------------------
print("\n========== CV Summary ==========")
for s in fold_summaries:
    print(
        f"Fold {s['fold']}: best val loss={s['best_val_loss']:.4f} | "
        f"best val acc={s['best_val_acc']:.4f} | "
        f"best val f1={s['best_val_f1']:.4f}"
    )

mean_acc = np.nanmean([s["best_val_acc"] for s in fold_summaries])
mean_f1  = np.nanmean([s["best_val_f1"]  for s in fold_summaries])
print(f"Mean(best val acc) across folds: {mean_acc:.4f}")
print(f"Mean(best val f1 ) across folds: {mean_f1:.4f}")


# -------------------------------------------------------------------
# CV Confusion Matrix
# -------------------------------------------------------------------
all_cv_preds   = np.array(all_cv_preds)
all_cv_targets = np.array(all_cv_targets)

cm_cv      = confusion_matrix(all_cv_targets, all_cv_preds)
cm_cv_norm = np.divide(
    cm_cv.astype("float"),
    cm_cv.sum(axis=1, keepdims=True),
    where=cm_cv.sum(axis=1, keepdims=True) != 0,
)

class_names = sorted(label_map.keys())  # Dynamic — adapts to whatever classes preprocessing produced

annot_cv = np.empty_like(cm_cv).astype(str)
for i in range(cm_cv.shape[0]):
    for j in range(cm_cv.shape[1]):
        count   = cm_cv[i, j]
        percent = cm_cv_norm[i, j] * 100
        annot_cv[i, j] = "0" if count == 0 else f"{count}\n({percent:.1f}%)"

plt.figure(figsize=(8, 6))
sns.heatmap(cm_cv, annot=annot_cv, fmt="", cmap="viridis",
            xticklabels=class_names, yticklabels=class_names, cbar=True)
plt.title("Confusion Matrix (Cross-Validation)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.show()


# -------------------------------------------------------------------
# Save best checkpoint
# -------------------------------------------------------------------
if best_fold_state is not None:
    torch.save(best_fold_state, checkpoint_path)
    print("Saved best-fold checkpoint to:", checkpoint_path)
else:
    print("WARNING: best_fold_state is None — no checkpoint saved.")

## 12. Training Curves (Last Fold)

In [ ]:
last = fold_summaries[-1]["history"]

plt.figure(figsize=(10, 5))
plt.plot(last["train_loss"], label="Train loss")
plt.plot(last["val_loss"],   label="Val/Test loss")
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.legend(); plt.title("Loss Curves"); plt.show()

plt.figure(figsize=(10, 5))
plt.plot(last["train_acc"], label="Train acc")
plt.plot(last["val_acc"],   label="Val/Test acc")
plt.xlabel("Epoch"); plt.ylabel("Accuracy")
plt.legend(); plt.title("Accuracy Curves"); plt.show()

plt.figure(figsize=(10, 5))
plt.plot(last["train_f1"], label="Train macro-F1")
plt.plot(last["val_f1"],   label="Val/Test macro-F1")
plt.xlabel("Epoch"); plt.ylabel("Macro-F1")
plt.legend(); plt.title("Macro-F1 Curves"); plt.show()

plt.figure(figsize=(10, 5))
plt.plot(last["val_w_cnn"],   label="Val attn w_cnn  (MFCC+Chroma CNN)")
plt.plot(last["val_w_lstm"],  label="Val attn w_lstm (MFCC+Chroma LSTM)")
plt.plot(last["val_w_inter"], label="Val attn w_inter (CNN+LSTM)")
plt.xlabel("Epoch"); plt.ylabel("Mean gate weight")
plt.legend(); plt.title("Attention/Gated Fusion Weight Trends (Validation)")
plt.show()

## 13. Unseen Test Set Evaluation

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Load best checkpoint
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.eval()

all_preds, all_targets = [], []

with torch.no_grad():
    for mfcc, chroma, y in unseen_loader:
        logits = model(mfcc.to(device), chroma.to(device))
        all_preds.append(torch.argmax(logits, dim=1).cpu().numpy())
        all_targets.append(y.numpy())

all_preds   = np.concatenate(all_preds)
all_targets = np.concatenate(all_targets)

acc  = accuracy_score(all_targets, all_preds)
prec, rec, f1, _ = precision_recall_fscore_support(
    all_targets, all_preds, average="macro", zero_division=0
)
print(f"Unseen Accuracy: {acc:.4f}")
print(f"Unseen Macro Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}")
print("\nClassification report:")
print(classification_report(all_targets, all_preds, target_names=class_names, digits=4))


cm      = confusion_matrix(all_targets, all_preds)
cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)

annot = np.empty_like(cm).astype(str)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        count   = cm[i, j]
        percent = cm_norm[i, j] * 100
        annot[i, j] = "0" if count == 0 else f"{count}\n({percent:.1f}%)"

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=annot, fmt="", cmap="viridis",
            xticklabels=class_names, yticklabels=class_names, cbar=True)
plt.title("Confusion Matrix (Unseen)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.show()

In [ ]:
# ── CV Summary Across Folds ──────────────────────────────────────────
import numpy as np

fold_accs  = [s["best_val_acc"] for s in fold_summaries]
fold_f1s   = [s["best_val_f1"]  for s in fold_summaries]
fold_losses = [s["best_val_loss"] for s in fold_summaries]

print("=" * 55)
print("CROSS-VALIDATION SUMMARY (10 Folds)")
print("=" * 55)
print(f"{'Metric':<25} {'Mean':>10} {'Std':>10}")
print("=" * 55)
print(f"{'Val Accuracy':<25} {np.mean(fold_accs):>10.4f} {np.std(fold_accs):>10.4f}")
print(f"{'Val Macro F1':<25} {np.mean(fold_f1s):>10.4f}  {np.std(fold_f1s):>10.4f}")
print(f"{'Val Loss':<25} {np.mean(fold_losses):>10.4f} {np.std(fold_losses):>10.4f}")
print("=" * 55)

print("\nPer-Fold Breakdown:")
for s in fold_summaries:
    print(f"  Fold {s['fold']:02d}: acc={s['best_val_acc']:.4f} | f1={s['best_val_f1']:.4f} | loss={s['best_val_loss']:.4f}")